# 🦴 Osteoporosis Prediction System
## Multi-Model Training Pipeline (XGBoost, SVM, Neural Network)

In [ ]:
import pandas as pd
import numpy as np
import pickle
import warnings
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.svm import SVC
from xgboost import XGBClassifier
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, BatchNormalization
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.optimizers import Adam

---
## 1. LOAD & PREPROCESS DATA

In [ ]:
def load_and_preprocess(filepath="osteoporosis.csv"): # Point to existing data
    df = pd.read_csv(filepath)
    df.drop(columns=["Id"], inplace=True, errors="ignore")

    # Fix 1: Changed fillna to 'Not Applicable'
    for col in ["Alcohol Consumption", "Medical Conditions", "Medications"]:
        if col in df.columns:
            df[col].fillna("Not Applicable", inplace=True)

    for col in df.select_dtypes(include="object").columns:
        if df[col].isnull().sum() > 0:
            df[col].fillna(df[col].mode()[0], inplace=True)

    # Fix 1: Drop 'data_source' before encoding
    if 'data_source' in df.columns:
        df.drop(columns=['data_source'], inplace=True)

    bins   = [0, 30, 45, 60, 75, 120]
    labels = ["Young Adult", "Adult", "Middle-Aged", "Senior", "Elderly"]
    df["Age_Group"] = pd.cut(df["Age"], bins=bins, labels=labels, right=False)
    df.drop(columns=["Age"], inplace=True)

    encoders = {}
    for col in df.select_dtypes(include=["object", "category"]).columns:
        le = LabelEncoder()
        df[col] = le.fit_transform(df[col].astype(str))
        encoders[col] = le
        # Fix 1: Print encoder classes for debugging
        print(f"Encoder for {col}: {le.classes_}")

    pickle.dump(encoders, open("encoders.pkl", "wb"))

    X = df.drop(columns=["Osteoporosis"])
    y = df["Osteoporosis"]

    print(f"✅ Dataset loaded: {X.shape[0]} rows, {X.shape[1]} features")
    return X, y


---
## 2. TRAIN-TEST SPLIT + SCALING

In [ ]:
def prepare_data(X, y):
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42, stratify=y
    )

    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled  = scaler.transform(X_test)

    # Fix 2: Save scaler inside function
    import pickle
    pickle.dump(scaler, open("scaler.pkl", "wb"))

    return X_train, X_test, y_train, y_test, X_train_scaled, X_test_scaled, scaler


---
## 3. MODEL 1 — XGBoost

In [ ]:
from sklearn.metrics import accuracy_score, f1_score, recall_score, roc_auc_score

def evaluate_model(name, y_test, y_pred, y_prob=None):
    # Fix 3: Comprehensive evaluation function
    acc = accuracy_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    recall = recall_score(y_test, y_pred)
    
    print(f"\n📊 Results for {name}:")
    print(f"   Accuracy: {acc*100:.2f}%")
    print(f"   F1 Score: {f1:.4f}")
    print(f"   Recall:   {recall:.4f}")
    
    if y_prob is not None:
        auc = roc_auc_score(y_test, y_prob)
        print(f"   ROC-AUC:  {auc:.4f}")
    
    return f1


In [ ]:
def train_xgboost(X_train, X_test, y_train, y_test):
    print("\n" + "="*50)
    print("🌲 Training XGBoost...")
    print("="*50)

    # Fix 4: Add scale_pos_weight to handle class imbalance
    scale = (y_train == 0).sum() / (y_train == 1).sum()
    
    model = XGBClassifier(
        n_estimators=300,
        max_depth=4,
        learning_rate=0.05,
        scale_pos_weight=scale, # Fix 4
        eval_metric="logloss",
        random_state=42,
        n_jobs=-1
    )
    model.fit(X_train, y_train)

    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:, 1]
    
    # Fix 3: Use evaluate_model and return F1
    f1 = evaluate_model("XGBoost", y_test, y_pred, y_prob)
    return model, f1


---
## 4. MODEL 2 — SVM

In [ ]:
def train_svm(X_train_scaled, X_test_scaled, y_train, y_test):
    print("\n" + "="*50)
    print("🔷 Training SVM...")
    print("="*50)

    # Fix 5: Add class_weight='balanced'
    model = SVC(
        kernel="rbf",
        C=10,
        class_weight="balanced", # Fix 5
        probability=True,
        random_state=42
    )
    model.fit(X_train_scaled, y_train)

    y_pred = model.predict(X_test_scaled)
    y_prob = model.predict_proba(X_test_scaled)[:, 1]
    
    # Fix 3: Use evaluate_model and return F1
    f1 = evaluate_model("SVM", y_test, y_pred, y_prob)
    return model, f1


---
## 5. MODEL 3 — NEURAL NETWORK
**14 Input | 5 Hidden Layers | 1 Output | 100 Epochs**

In [ ]:
def build_neural_network(input_dim=14):
    model = Sequential([

        # Hidden Layer 1
        Dense(128, input_dim=input_dim, activation="relu"),  # ✅ 256 → 128
        BatchNormalization(),
        Dropout(0.3),

        # Hidden Layer 2
        Dense(64, activation="relu"),                         # ✅ 128 → 64
        BatchNormalization(),
        Dropout(0.3),

        # Hidden Layer 3
        Dense(32, activation="relu"),                         # ✅ 64 → 32
        BatchNormalization(),
        Dropout(0.2),

        # ✅ REMOVED: Layer 4 aur Layer 5

        # Output Layer
        Dense(1, activation="sigmoid")
    ])

    model.compile(
        optimizer=Adam(learning_rate=0.001),
        loss="binary_crossentropy",
        metrics=["accuracy"]
    )

    return model

In [ ]:
from sklearn.utils.class_weight import compute_class_weight

def train_neural_network(X_train_scaled, X_test_scaled, y_train, y_test):
    print("\n" + "="*50)
    print("🧠 Training Neural Network...")
    print("="*50)

    # Fix 6: Calculate class weights for Neural Network
    weights = compute_class_weight("balanced", classes=np.unique(y_train), y=y_train)
    class_weight_dict = dict(enumerate(weights))

    input_dim = X_train_scaled.shape[1]
    model = build_neural_network(input_dim=input_dim)

    model.fit(
        X_train_scaled, y_train,
        epochs=100,
        batch_size=64,
        class_weight=class_weight_dict, # Fix 6
        validation_split=0.15,
        verbose=0
    )

    y_prob = model.predict(X_test_scaled).flatten()
    y_pred = (y_prob > 0.5).astype(int)
    
    # Fix 3: Use evaluate_model and return F1
    f1 = evaluate_model("Neural Network", y_test, y_pred, y_prob)
    return model, f1


---
## 6. AUTO SELECT BEST MODEL

In [ ]:
def select_best_model(models_dict):
    print("\n" + "="*50)
    print("🏆 MODEL COMPARISON (F1 SCORE)")
    print("="*50)

    # Fix 7: Compare using F1 Score
    for name, (model, f1) in models_dict.items():
        print(f"  {name:20s} → F1 Score: {f1:.4f}")

    best_name = max(models_dict, key=lambda k: models_dict[k][1])
    best_model, best_f1 = models_dict[best_name]

    print(f"\n🥇 BEST MODEL: {best_name} with {best_f1:.4f} F1 Score")
    print("="*50)
    return best_name, best_model, best_f1


---
## 7. SAVE MODELS & BEST MODEL INFO

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

def analyze_body_weight(filepath="osteoporosis_with_overweight.csv"):
    # Fix 8: Analyze osteoporosis rate by Body Weight and data source
    df = pd.read_csv(filepath)
    
    print("\n⚖️ Body Weight Analysis (Real vs Synthetic)")
    analysis = df.groupby(['data_source', 'Body Weight'])['Osteoporosis'].mean().unstack()
    print(analysis)
    
    plt.figure(figsize=(10, 6))
    analysis.plot(kind='bar')
    plt.title("Osteoporosis Rate by Body Weight (Real vs Synthetic)")
    plt.ylabel("Osteoporosis Rate")
    plt.show()


In [ ]:
def save_models(xgb_model, svm_model, nn_model, scaler, best_name):
    pickle.dump(xgb_model, open("xgboost_model.pkl", "wb"))
    pickle.dump(svm_model,  open("svm_model.pkl", "wb"))
    pickle.dump(scaler,     open("scaler.pkl", "wb"))
    nn_model.save("neural_network_model.h5")

    # Save best model name
    with open("best_model.txt", "w") as f:
        f.write(best_name)

    print("\n💾 All models saved successfully!")
    print(f"   Best model recorded: {best_name}")

---
## 8. PREDICTION FUNCTION (uses best model)

In [ ]:
def predict_osteoporosis(input_data: dict):
    # Fix 9: Updates to prediction function
    with open("best_model.txt", "r") as f:
        best_name = f.read().strip()

    scaler   = pickle.load(open("scaler.pkl",   "rb"))
    encoders = pickle.load(open("encoders.pkl", "rb"))

    df_input = pd.DataFrame([input_data])
    # Fix 9: Handle 'None' values specifically
    df_input.replace("None", "Not Applicable", inplace=True)
    df_input.fillna("Not Applicable", inplace=True)

    if "Age" in df_input.columns:
        bins   = [0, 30, 45, 60, 75, 120]
        labels = ["Young Adult", "Adult", "Middle-Aged", "Senior", "Elderly"]
        df_input["Age_Group"] = pd.cut(df_input["Age"], bins=bins, labels=labels, right=False)
        df_input.drop(columns=["Age"], inplace=True)

    for col, le in encoders.items():
        if col in df_input.columns:
            val = str(df_input[col].iloc[0])
            # Fix 9: Unknown value protection
            if val not in le.classes_:
                val = le.classes_[0]
            df_input[col] = le.transform([val])[0]

    scaled_input = scaler.transform(df_input)

    if best_name == "XGBoost":
        model = pickle.load(open("xgboost_model.pkl", "rb"))
        pred = model.predict(df_input)[0]
        prob = model.predict_proba(df_input)[0][1]
    elif best_name == "SVM":
        model = pickle.load(open("svm_model.pkl", "rb"))
        pred = model.predict(scaled_input)[0]
        prob = model.predict_proba(scaled_input)[0][1]
    elif best_name == "Neural Network":
        model = tf.keras.models.load_model("neural_network_model.h5")
        # Fix 9: Use verbose=0 to hide progress bar
        prob = float(model.predict(scaled_input, verbose=0)[0][0])
        pred = int(prob > 0.5)

    return "Osteoporosis Detected ⚠️" if pred == 1 else "No Osteoporosis ✅"


---
## 9. MAIN — TRAIN ALL & SELECT BEST

In [ ]:
import os

# Fix 10: Existence check before training
if os.path.exists("best_model.txt") and os.path.exists("xgboost_model.pkl"):
    print("✅ Models already saved, go to Cell 13 for prediction.")
else:
    print("\n🦴 OSTEOPOROSIS PREDICTION SYSTEM — STARTING TRAINING\n")
    X, y = load_and_preprocess("osteoporosis_with_overweight.csv")
    analyze_body_weight("osteoporosis_with_overweight.csv") # Fix 8 call
    
    X_train, X_test, y_train, y_test, X_train_scaled, X_test_scaled, scaler = prepare_data(X, y)
    
    xgb_model, xgb_f1 = train_xgboost(X_train, X_test, y_train, y_test)
    svm_model, svm_f1 = train_svm(X_train_scaled, X_test_scaled, y_train, y_test)
    nn_model, nn_f1 = train_neural_network(X_train_scaled, X_test_scaled, y_train, y_test)
    
    models_dict = {
        "XGBoost": (xgb_model, xgb_f1),
        "SVM": (svm_model, svm_f1),
        "Neural Network": (nn_model, nn_f1)
    }
    
    best_name, best_model, best_f1 = select_best_model(models_dict)
    save_models(xgb_model, svm_model, nn_model, scaler, best_name)



🦴 OSTEOPOROSIS PREDICTION SYSTEM — STARTING TRAINING

✅ Dataset loaded: 21958 rows, 14 features
   Target classes: {0: 12952, 1: 9006}


C:\Users\Sahil Gangurde\AppData\Local\Temp\ipykernel_16232\1398331912.py:7: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df[col].fillna("Unknown", inplace=True)


In [ ]:
# Step 2: Prepare splits
X_train, X_test, y_train, y_test, \
X_train_scaled, X_test_scaled, scaler = prepare_data(X, y)

In [ ]:
# Step 3a: Train XGBoost
xgb_model, xgb_acc = train_xgboost(X_train, X_test, y_train, y_test)


🌲 Training XGBoost...
✅ XGBoost Accuracy: 97.86%
              precision    recall  f1-score   support

           0       0.98      0.99      0.98      2591
           1       0.98      0.97      0.97      1801

    accuracy                           0.98      4392
   macro avg       0.98      0.98      0.98      4392
weighted avg       0.98      0.98      0.98      4392



In [ ]:
# Step 3b: Train SVM
svm_model, svm_acc = train_svm(X_train_scaled, X_test_scaled, y_train, y_test)


🔷 Training SVM...
✅ SVM Accuracy: 97.59%
              precision    recall  f1-score   support

           0       0.98      0.98      0.98      2591
           1       0.97      0.97      0.97      1801

    accuracy                           0.98      4392
   macro avg       0.98      0.97      0.98      4392
weighted avg       0.98      0.98      0.98      4392



In [ ]:
# Step 3c: Train Neural Network
nn_model, nn_acc, history = train_neural_network(X_train_scaled, X_test_scaled, y_train, y_test)


🧠 Training Neural Network (5 Hidden Layers, 100 Epochs)...


c:\Users\Sahil Gangurde\AppData\Local\Programs\Python\Python310\lib\site-packages\keras\src\layers\core\dense.py:95: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 128)            │         1,920 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 128)            │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 64)             │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_2           │ (None, 32)             │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 13,185 (51.50 KB)

 Trainable params: 12,737 (49.75 KB)

 Non-trainable params: 448 (1.75 KB)

Epoch 1/100
234/234 ━━━━━━━━━━━━━━━━━━━━ 8s 9ms/step - accuracy: 0.9026 - loss: 0.2686 - val_accuracy: 0.9594 - val_loss: 0.1538 - learning_rate: 0.0010
Epoch 2/100
234/234 ━━━━━━━━━━━━━━━━━━━━ 2s 7ms/step - accuracy: 0.9499 - loss: 0.1687 - val_accuracy: 0.9628 - val_loss: 0.1273 - learning_rate: 0.0010
Epoch 3/100
234/234 ━━━━━━━━━━━━━━━━━━━━ 2s 6ms/step - accuracy: 0.9573 - loss: 0.1414 - val_accuracy: 0.9643 - val_loss: 0.1113 - learning_rate: 0.0010
Epoch 4/100
234/234 ━━━━━━━━━━━━━━━━━━━━ 2s 7ms/step - accuracy: 0.9636 - loss: 0.1243 - val_accuracy: 0.9704 - val_loss: 0.0935 - learning_rate: 0.0010
Epoch 5/100
234/234 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - accuracy: 0.9670 - loss: 0.1153 - val_accuracy: 0.9738 - val_loss: 0.0875 - learning_rate: 0.0010
Epoch 6/100
234/234 ━━━━━━━━━━━━━━━━━━━━ 2s 6ms/step - accuracy: 0.9682 - loss: 0.1110 - val_accuracy: 0.9742 - val_loss: 0.0892 - learning_rate: 0.0010
Epoch 7/100
234/234 ━━━━━━━━━━━━━━━━━━━━ 3s 8ms/step - accuracy: 0.9680 - loss: 0.

In [ ]:
# Step 4: Compare & select best
models_dict = {
    "XGBoost":        (xgb_model, xgb_acc),
    "SVM":            (svm_model, svm_acc),
    "Neural Network": (nn_model,  nn_acc),
}

best_name, best_model, best_acc = select_best_model(models_dict)


🏆 MODEL COMPARISON
  XGBoost              → Accuracy: 97.86%
  SVM                  → Accuracy: 97.59%
  Neural Network       → Accuracy: 97.91%

🥇 BEST MODEL: Neural Network with 97.91% accuracy


In [ ]:
# Step 5: Save everything
save_models(xgb_model, svm_model, nn_model, scaler, best_name)


💾 All models saved successfully!
   Best model recorded: Neural Network


---
## 🧪 SAMPLE PREDICTION TEST

In [ ]:
sample_input = {
    "Age": 50,
    "Gender": "Female",
    "Hormonal Changes": "Postmenopausal",
    "Family History": "Yes",
    "Race/Ethnicity": "African American",
    "Body Weight": "Normal",
    "Calcium Intake": "Low",
    "Vitamin D Intake": "Sufficient",
    "Physical Activity": "Active",
    "Smoking": "No",
    "Alcohol Consumption": "None",
    "Medical Conditions": "None",
    "Medications": "None",
    "Prior Fractures": "No"
}
result = predict_osteoporosis(sample_input)
print(f"\n📊 Final Result: {result}")

ValueError: y contains previously unseen labels: 'None'

---
## 10. CLINICAL PERFORMANCE EVALUATION
**Generating Visual Evidence for Real-World Reliability**

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix, roc_curve, auc
import numpy as np

def run_clinical_evaluation(model, X_test, y_test, model_name):
    print(f"\n📊 Generating Performance Report for {model_name}...")
    
    # 1. Predictions
    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:, 1]
    
    # 2. Confusion Matrix
    cm = confusion_matrix(y_test, y_pred)
    plt.figure(figsize=(15, 5))
    
    plt.subplot(1, 3, 1)
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", cbar=False)
    plt.title(f"Confusion Matrix: {model_name}")
    plt.xlabel("Predicted (0=Low, 1=High)")
    plt.ylabel("Actual (0=Low, 1=High)")
    
    # 3. ROC Curve
    fpr, tpr, _ = roc_curve(y_test, y_prob)
    roc_auc = auc(fpr, tpr)
    
    plt.subplot(1, 3, 2)
    plt.plot(fpr, tpr, color="darkorange", lw=2, label=f"ROC curve (AUC = {roc_auc:.2f})")
    plt.plot([0, 1], [0, 1], color="navy", lw=2, linestyle="--")
    plt.title(f"ROC Curve: {model_name}")
    plt.xlabel("False Positive Rate")
    plt.ylabel("True Positive Rate")
    plt.legend(loc="lower right")
    
    # 4. Feature Importance (Specific to XGBoost)
    if model_name == "XGBoost":
        plt.subplot(1, 3, 3)
        importances = model.feature_importances_
        indices = np.argsort(importances)[-10:] # Top 10 factors
        plt.barh(range(len(indices)), importances[indices], align="center", color='#4A3AFF')
        plt.yticks(range(len(indices)), [X_test.columns[i] for i in indices])
        plt.title("Top 10 Risk Factors (Logic Check)")
        plt.xlabel("Predictive Power")
    
    plt.tight_layout()
    plt.show()

# Running evaluation for the best model
if 'xgb_model' in globals():
    run_clinical_evaluation(xgb_model, X_test, y_test, "XGBoost")
else:
    print("❌ Error: 'xgb_model' not found. Please run the training cells first.")


---
## 11. EXPLAINABLE AI (SHAP)
**Understanding Individual Patient Risk Drivers**

In [ ]:
import shap

def explain_prediction_with_shap(model, X_sample, model_name):
    print(f"\n🔍 Explaining prediction for a sample patient using SHAP...")
    
    # SHAP supports Tree models (XGBoost) natively and very fast
    if model_name == "XGBoost":
        explainer = shap.TreeExplainer(model)
        shap_values = explainer.shap_values(X_sample)
        
        # Visualize the first prediction in the sample
        # Waterfall plot shows how each feature pushed the risk up or down
        shap.plots.waterfall(shap.Explanation(values=shap_values[0], 
                                              base_values=explainer.expected_value, 
                                              data=X_sample.iloc[0], 
                                              feature_names=X_sample.columns))
    else:
        print("SHAP visualization in this notebook is currently optimized for XGBoost.")

# Let's explain the first person in our test set
if 'xgb_model' in globals():
    explain_prediction_with_shap(xgb_model, X_test.head(1), "XGBoost")
